# Concept Normalization System
## Board Game Ontology Entity Linking via Local LLM (Ollama)

This notebook implements a **two-pass NLP pipeline** that maps ontology concepts to their occurrences in free text.

### Pipeline

```
Ontology (TTL)
      ↓ rdflib.parse()
  ontology dict (classes / instances / properties)
      ↓ build_ontology_context()
  static ontology reference text  →  added to user message in each API call

  Input text
      ↓
  [Pass 1] qwen2.5:14b via Ollama  (system = PASS1_SYSTEM)
  → JSON: mentions[] with span + candidate_concept + rationale
      ↓ find_offsets()  — char offsets resolved in Python
  [Pass 2] qwen2.5:14b × N  (system = PASS2_SYSTEM, one call per mention)
  → JSON per mention: concept + concept_uri + instance + instance_uri + confidence + reasoning
      ↓ filter(confidence ≥ 0.3, concept ≠ "NoMatch")
  Annotations  →  HTML highlight  +  pandas table
```

**Ontology:** `BoardGameOntology-0MI3400805.ttl`  
**Model:** `qwen2.5:14b` running locally via [Ollama](https://ollama.com)

In [ ]:
from __future__ import annotations

import json
import textwrap
from pathlib import Path
from collections import defaultdict
from importlib.metadata import version as pkg_version

import ollama
import pandas as pd
from rdflib import Graph, RDF, RDFS, OWL
from IPython.display import display, HTML

# --- Configuration ---
_TTL_NAME    = "BoardGameOntology-0MI3400805.ttl"
_search_dirs = [Path().resolve()] + [d for d in Path().resolve().iterdir() if d.is_dir()]
_NOTEBOOK_DIR = next((d for d in _search_dirs if (d / _TTL_NAME).exists()), Path().resolve())

ONTOLOGY_PATH = _NOTEBOOK_DIR / _TTL_NAME
OLLAMA_MODEL  = "qwen2.5:14b"
BGO_BASE      = "http://www.semanticweb.org/nikolay/ontologies/2025/11/board-game-ontology/"

# Colors for HTML highlighting — unlisted concepts fall back to #7f8c8d (grey)
CONCEPT_COLORS = {
    "BoardGame":           "#2980b9",
    "CompetitiveGame":     "#1a6fa8",
    "CooperativeGame":     "#1a5276",
    "SemiCooperativeGame": "#154360",
    "SoloFriendlyGame":    "#1f618d",
    "Expansion":           "#5dade2",
    "GameMechanics":       "#c0392b",
    "GameCategory":        "#e67e22",
    "GameType":            "#f39c12",
    "Designer":            "#8e44ad",
    "Publisher":           "#27ae60",
    "Person":              "#6c3483",
    # Data properties — teal family
    "playerCount":         "#16a085",
    "minCount":            "#16a085",
    "maxCount":            "#16a085",
    "playingTime":         "#0e6655",
    "minPlayingTime":      "#0e6655",
    "maxPlayingTime":      "#0e6655",
    "complexity":          "#795548",
    "rating":              "#e65100",
    "releaseYear":         "#546e7a",
    "suggestedAge":        "#6d4c41",
}

print(f"Configuration OK — model: {OLLAMA_MODEL}")
print(f"Ontology path:    {ONTOLOGY_PATH}  (exists: {ONTOLOGY_PATH.exists()})")
print(f"Ollama library:   {pkg_version('ollama')}")

In [ ]:
def load_ontology(ttl_path: Path) -> dict:
    """Parse the TTL ontology and return structured dicts for classes, properties, and instances."""
    g = Graph()
    g.parse(str(ttl_path), format="turtle")

    classes = []
    for uri in g.subjects(RDF.type, OWL.Class):
        if not str(uri).startswith(BGO_BASE):
            continue
        local = str(uri).split("/")[-1]
        label = g.value(uri, RDFS.label)
        parents = [
            str(p).split("/")[-1]
            for p in g.objects(uri, RDFS.subClassOf)
            if str(p).startswith(BGO_BASE)
        ]
        classes.append({
            "uri": str(uri), "local": local,
            "label": str(label) if label else local,
            "subClassOf": parents,
        })

    obj_props = []
    for uri in g.subjects(RDF.type, OWL.ObjectProperty):
        if not str(uri).startswith(BGO_BASE):
            continue
        local = str(uri).split("/")[-1]
        domain = [str(d).split("/")[-1] for d in g.objects(uri, RDFS.domain) if str(d).startswith(BGO_BASE)]
        range_ = [str(r).split("/")[-1] for r in g.objects(uri, RDFS.range)  if str(r).startswith(BGO_BASE)]
        obj_props.append({"uri": str(uri), "local": local, "domain": domain, "range": range_})

    data_props = []
    for uri in g.subjects(RDF.type, OWL.DatatypeProperty):
        if not str(uri).startswith(BGO_BASE):
            continue
        local = str(uri).split("/")[-1]
        domain = [str(d).split("/")[-1] for d in g.objects(uri, RDFS.domain) if str(d).startswith(BGO_BASE)]
        data_props.append({"uri": str(uri), "local": local, "domain": domain})

    bgo_class_uris = {str(c["uri"]) for c in classes}
    instances = []
    for s in g.subjects(RDF.type, OWL.NamedIndividual):
        if not str(s).startswith(BGO_BASE):
            continue
        local = str(s).split("/")[-1]
        label = g.value(s, RDFS.label)
        types = [
            str(t).split("/")[-1]
            for t in g.objects(s, RDF.type)
            if str(t).startswith(BGO_BASE) and str(t) in bgo_class_uris
        ]
        instances.append({
            "uri": str(s), "local": local,
            "label": str(label) if label else local,
            "types": types,
        })

    return {
        "classes":    classes,
        "obj_props":  obj_props,
        "data_props": data_props,
        "instances":  instances,
    }


ontology = load_ontology(ONTOLOGY_PATH)
print(f"Classes:           {len(ontology['classes'])}")
print(f"Object Properties: {len(ontology['obj_props'])}")
print(f"Data Properties:   {len(ontology['data_props'])}")
print(f"Instances:         {len(ontology['instances'])}")

# Derived lookups — built once, used throughout
_class_names  = [c["local"] for c in sorted(ontology["classes"],    key=lambda x: x["local"])]
_dprop_names  = [p["local"] for p in sorted(ontology["data_props"], key=lambda x: x["local"])]
_CONCEPT_ENUM = _class_names + _dprop_names + ["NoMatch"]

# Class → direct instance locals (for dynamic Pass 2 schema)
_instances_by_class: dict = defaultdict(list)
for _inst in ontology["instances"]:
    for _t in _inst["types"]:
        _instances_by_class[_t].append(_inst["local"])

# Pass 1 schema — mention_type separates CLASS from DATA_PROPERTY mentions
PASS1_SCHEMA = {
    "type": "object",
    "properties": {
        "mentions": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "span":              {"type": "string"},
                    "mention_type":      {"type": "string", "enum": ["CLASS", "DATA_PROPERTY"]},
                    "candidate_concept": {"type": "string", "enum": _CONCEPT_ENUM},
                    "rationale":         {"type": "string"},
                },
                "required": ["span", "mention_type", "candidate_concept", "rationale"],
            },
        }
    },
    "required": ["mentions"],
}
# Pass 2 schema is generated dynamically per mention — see _make_pass2_schema()

print(f"\nConcept enum ({len(_CONCEPT_ENUM)} values):")
print(f"  Classes ({len(_class_names)}):    {', '.join(_class_names)}")
print(f"  Data props ({len(_dprop_names)}): {', '.join(_dprop_names)}")

In [ ]:
def build_ontology_context(ontology: dict) -> str:
    """Build the static ontology reference text sent in the user message of each API call."""
    lines = []
    lines.append("## Board Game Ontology Reference\n")
    lines.append(f"Prefix `bgo:` = `{BGO_BASE}`\n")

    lines.append("### CLASSES")
    for c in sorted(ontology["classes"], key=lambda x: x["local"]):
        parent_str = (f" (subClassOf: {', '.join(c['subClassOf'])})" if c["subClassOf"] else "")
        lines.append(f"- `bgo:{c['local']}`{parent_str}")
    lines.append("")

    lines.append("### OBJECT PROPERTIES")
    for p in sorted(ontology["obj_props"], key=lambda x: x["local"]):
        dom = ", ".join(p["domain"]) or "unspecified"
        rng = ", ".join(p["range"])  or "unspecified"
        lines.append(f"- `bgo:{p['local']}` (domain: {dom} → range: {rng})")
    lines.append("")

    lines.append("### DATA PROPERTIES")
    for p in sorted(ontology["data_props"], key=lambda x: x["local"]):
        dom = ", ".join(p["domain"]) or "unspecified"
        lines.append(f"- `bgo:{p['local']}` (domain: {dom})")
    lines.append("")

    lines.append("### NAMED INDIVIDUALS")
    by_type: dict = defaultdict(list)
    for inst in ontology["instances"]:
        primary_type = inst["types"][0] if inst["types"] else "Other"
        by_type[primary_type].append(inst)

    for type_name in sorted(by_type.keys()):
        lines.append(f"\n#### {type_name}")
        for inst in sorted(by_type[type_name], key=lambda x: x["label"]):
            label_str = (f' (label: "{inst["label"]}")' if inst["label"] != inst["local"] else "")
            lines.append(f"- `bgo:{inst['local']}`{label_str}")
    lines.append("")

    return "\n".join(lines)


ontology_context = build_ontology_context(ontology)
print(f"Ontology context: {len(ontology_context)} characters")
print("\n--- Preview (first 800 chars) ---")
print(ontology_context[:800])

In [4]:
# --- Prompt templates (ontology context goes in user message, not here) ---

PASS1_SYSTEM = textwrap.dedent("""\
    You are a strict, high-precision Named Entity Recognition (NER) system for board game reviews, rules, and descriptions.
    Your task is to scan the provided text and extract text spans (mentions) that map directly to the categories or attributes defined in the Board Game Ontology.
    
    CRITICAL RULES FOR EXTRACTION:
    1. A span must be exact. Do not include surrounding punctuation or generic adjectives (e.g., extract "Hand Management", NOT "excellent hand management").
    2. Split mentions when appropriate. If text says "Klaus Teuber and Matt Leacock", extract them as TWO separate mentions of type "CLASS" (Designer).
    3. Categorize every extracted span strictly into one of two categories:
       - "CLASS": For real-world named entities, mechanics, types, or categories (e.g., game titles, designer names, publisher studios, mechanic terms).
       - "DATA_PROPERTY": For raw values, numbers, or factual attributes (e.g., "45 minutes", "2004", "14+", "8.5").
    4. Do NOT restrict extraction only to the games or mechanics listed in the ontology reference. If the text mentions a board game title (like "Ark Nova") or a game mechanic (like "area control" or "worker placement"), you MUST extract it as a CLASS mention, even if that specific entity is missing from the ontology individuals.                          
    
    DATA_PROPERTY PRIORITY RULES:
    - If a span indicates a specific number of players (e.g., "4 players"), map to 'playerCount' or its subproperties ('minCount'/'maxCount') based on ontology definitions.
    - If a span indicates duration (e.g., "60-90 mins"), map to 'playingTime' or its subproperties.
    - Try to isolate the core value where possible (e.g., instead of "sitting at 7.4 rating", extract just "7.4").
    
    
    NEGATIVE CONSTRAINTS (DO NOT EXTRACT):
    - Game-internal components: Specific card names, resource tokens (e.g., "wood", "sheep"), character names, or faction names.
    - Generic descriptions: Words like "fun", "mechanics", "board game", "strategy" unless they function as an exact concept name.
    - Standalone numbers without explicit context (e.g., "turn 3", "roll a 6", "draw 2 cards").
    
    OUTPUT FORMAT REQUIREMENTS:
    You must output a JSON object adhering exactly to the requested schema. The 'candidate_concept' field MUST be chosen strictly from the list of classes and data properties available in the appended ontology reference.
""")

PASS2_SYSTEM = textwrap.dedent("""\
    You are an expert Semantic Web Concept Normalization and Entity Linking system.
    Your job is to take a single text span (mention), its context, and a candidate concept hint from Pass 1, and resolve it perfectly against the provided Board Game Ontology.
    
    RESOLUTION LOGIC:
    
    STEP 1: CONCEPT RESOLUTION
    - Look at the text span, context, and the hint. Choose the MOST SPECIFIC valid concept (Class or Data Property name) from the ontology reference.
    - If 'mention_type' is "CLASS", the concept must be an ontology CLASS (e.g., "CooperativeGame" instead of just "BoardGame" if the ontology specifies it).
    - If 'mention_type' is "DATA_PROPERTY", the concept must be an exact DATA PROPERTY name (e.g., "releaseYear", "suggestedAge", "minPlayingTime").
    - If the span has absolutely no relevance to any class or property in the ontology, return "NoMatch" for the concept.
    - Pay extreme attention to the context clues (syntax and capitalization). If a text span uses capitalized words and is described as a "prototype", "title", "box", or "session" (e.g., "we played a session of Card Draw"), it is a BoardGame instance, NOT a GameMechanics concept, regardless of what the words literally say.                           
    
    STEP 2: INSTANCE RESOLUTION (Only applicable if concept is a CLASS)
    - CRITICAL RULE (HIGHEST PRIORITY): You MUST thoroughly scan the "NAMED INDIVIDUALS" section of the ontology under the resolved class BEFORE deciding an instance is "NEW".
    - If the mention matches a listed individual (case-insensitive, ignoring spaces, underscores, colons, or punctuation, e.g., "Pandemic" matches "bgo:Pandemic", "Unstable Unicorns" matches "bgo:Unstable_Unicorns", "Klaus Teuber" matches "bgo:Klaus_Teuber"), you MUST return its exact local name identifier (e.g., "Pandemic", "Unstable_Unicorns", "Klaus_Teuber").
    - Absolute Constraint: Do NOT return "NEW" if the word or entity exists as a Named Individual in the provided ontology reference. "NEW" is a last resort.
    - Return "NEW" ONLY when you are 100% certain that the specific real-world entity is completely missing from the appended ontology text (e.g., "Ark Nova").
    - If the span refers to a generic concept rather than a specific individual (e.g., the span is "hand management" which maps to the concept class "GameMechanics"), return "none".
    - If the concept is a DATA_PROPERTY, always return "none" for the instance.
    - CRITICAL RECONCILIATION RULE: Do NOT force a match with a listed individual if the names are completely different (e.g., "Lookout Spiele" does NOT match "Breaking_Games" just because both are publishers). If there is no clear textual or semantic overlap in the name, you MUST return "NEW". 
    - Only return a listed individual if you are highly confident it is the exact same entity under a slightly different spelling.                             
    
    GOLDEN EXAMPLES FOR LOGIC:
    - Context: "We played >>> Pandemic <<< last night." 
      -> concept: "CooperativeGame", instance: "Pandemic" (Matches listed individual perfectly)
    - Context: "I love playing >>> Ark Nova <<<." 
      -> concept: "BoardGame", instance: "NEW" (Real game, but completely missing from ontology)
    - Context: "This game heavily relies on >>> hand management <<<." 
      -> concept: "GameMechanics", instance: "none" (It's a generic mechanic class, not a specific instance)
    - Context: "Designed for >>> 12+ <<< age group." 
      -> concept: "suggestedAge", instance: "none" (Data property)
    
    Strictly follow the JSON schema provided. Do not invent any properties or fields outside the schema.
""")

print(f"Pass 1 system prompt: {len(PASS1_SYSTEM):,} chars  (ontology added per-call in user message)")
print(f"Pass 2 system prompt: {len(PASS2_SYSTEM):,} chars  (ontology added per-call in user message)")

Pass 1 system prompt: 2,339 chars  (ontology added per-call in user message)
Pass 2 system prompt: 3,525 chars  (ontology added per-call in user message)


In [5]:
def find_offsets(text: str, span: str) -> tuple[int, int]:
    """Exact match first, then case-insensitive. Returns (0, 0) if not found."""
    idx = text.find(span)
    if idx != -1:
        return idx, idx + len(span)
    idx = text.lower().find(span.lower())
    if idx != -1:
        return idx, idx + len(span)
    return 0, 0


def _make_uri(local_name: str | None) -> str | None:
    """Reconstruct a full BGO URI from a local name. URIs never come from the model."""
    if not local_name or local_name in ("none", "NEW", "NoMatch", ""):
        return None
    return BGO_BASE + local_name


def _subclass_closure(class_name: str) -> set:
    """Return class_name and all its subclasses (recursive)."""
    result = {class_name}
    for c in ontology["classes"]:
        if class_name in c["subClassOf"]:
            result |= _subclass_closure(c["local"])
    return result


def _make_pass2_schema(hint_concept: str, mention_type: str) -> dict:
    """
    Build a Pass 2 JSON schema scoped to the hinted concept.
    - DATA_PROPERTY hint  → concept enum = data prop names only; instance forced to "none"
    - CLASS hint          → concept enum = all class names;
                            instance enum = instances of that class family + NEW/none
    """
    if mention_type == "DATA_PROPERTY":
        concept_enum  = _dprop_names + ["NoMatch"]
        instance_enum = ["none"]
    else:
        concept_enum = _class_names + ["NoMatch"]
        class_set    = _subclass_closure(hint_concept) if hint_concept in set(_class_names) else {hint_concept}
        instance_enum = sorted({
            local
            for cls in class_set
            for local in _instances_by_class.get(cls, [])
        }) + ["NEW", "none"]

    return {
        "type": "object",
        "properties": {
            "concept":    {"type": "string", "enum": concept_enum},
            "instance":   {"type": "string", "enum": instance_enum},
            "confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
            "reasoning":  {"type": "string"},
        },
        "required": ["concept", "instance", "confidence", "reasoning"],
    }

In [6]:
def pass1_extract_mentions(text: str, document_label: str = "document") -> list:
    """
    Pass 1: extract candidate spans with mention_type (CLASS or DATA_PROPERTY).
    Ontology context goes in the user message to preserve model attention on instructions.
    Offsets are resolved in Python — never asked from the model.
    """
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": PASS1_SYSTEM},
            {
                "role": "user",
                "content": (
                    f"Ontology Reference:\n{ontology_context}\n\n"
                    f"Extract all ontology-related mentions from the document below.\n\n"
                    f"Document ({document_label}):\n{text}"
                ),
            },
        ],
        format=PASS1_SCHEMA,
        options={"temperature": 0.0},
    )

    raw      = response.message.content or ""
    result   = json.loads(raw)
    mentions = result.get("mentions", [])
    print(f"  Pass 1 — {len(mentions)} raw mentions from model")

    resolved   = []
    seen_spans: set = set()
    for m in mentions:
        span = m.get("span", "").strip()
        if not span or span.lower() in seen_spans:
            continue
        seen_spans.add(span.lower())
        start, end = find_offsets(text, span)
        if start == end == 0 and span.lower() not in text.lower():
            continue
        m["span"]       = text[start:end] if start != end else span
        m["start_char"] = start
        m["end_char"]   = end
        # Normalise mention_type in case model omits it
        if "mention_type" not in m:
            m["mention_type"] = (
                "DATA_PROPERTY" if m.get("candidate_concept") in set(_dprop_names)
                else "CLASS"
            )
        resolved.append(m)

    print(f"  Pass 1: {len(resolved)} candidate mentions found")
    return resolved


def pass2_map_mention(mention: dict, text: str, context_window: int = 150) -> dict:
    """
    Pass 2: map span to ontology CONCEPT + optional instance.
    - Dynamic schema scoped to hint (reduces hallucination within enum).
    - Ontology context in user message (better Qwen attention).
    - URIs reconstructed in Python (model never generates URLs).
    - temperature=0.0 for deterministic output.
    """
    span  = mention.get("span", "")
    start = mention.get("start_char", 0)
    end   = mention.get("end_char", start + len(span))

    ctx_start   = max(0, start - context_window)
    ctx_end     = min(len(text), end + context_window)
    ctx         = text[ctx_start:ctx_end]
    local_s     = start - ctx_start
    local_e     = end   - ctx_start
    highlighted = ctx[:local_s] + ">>>" + ctx[local_s:local_e] + "<<<" + ctx[local_e:]

    hint         = mention.get("candidate_concept", "NoMatch")
    mention_type = mention.get("mention_type", "CLASS")
    schema       = _make_pass2_schema(hint, mention_type)

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": PASS2_SYSTEM},
            {
                "role": "user",
                "content": (
                    f"Ontology Reference:\n{ontology_context}\n\n"
                    f"Map this mention:\n"
                    f"Mention:      \"{span}\"\n"
                    f"mention_type: {mention_type}\n"
                    f"Hint:         {hint}\n"
                    f"Context (>>> marks the span <<<):\n{highlighted}"
                ),
            },
        ],
        format=schema,
        options={"temperature": 0.0},
    )

    mapping  = json.loads(response.message.content or "{}")

    # Normalise "none" string sentinel → Python None
    if mapping.get("instance") in ("none", ""):
        mapping["instance"] = None

    # Reconstruct URIs in Python — model never generates them
    mapping["concept_uri"]  = _make_uri(mapping.get("concept"))
    mapping["instance_uri"] = _make_uri(mapping.get("instance"))

    return {**mention, **mapping}


def run_two_pass_pipeline(
    text: str,
    document_label: str = "document",
    confidence_threshold: float = 0.3,
) -> list:
    """Full pipeline: Pass 1 (extraction) + Pass 2 (concept normalization)."""
    print(f"\n{'='*60}")
    print(f"Processing: {document_label}")
    print(f"{'='*60}")

    if not text.strip():
        print("  Empty document — skipping.")
        return []

    print("[ Pass 1 ] Extracting candidate mentions...")
    candidates = pass1_extract_mentions(text, document_label)

    if not candidates:
        print("  No candidates returned.")
        return []

    print(f"\n[ Pass 2 ] Concept normalization for {len(candidates)} mentions...")
    annotations = []
    for i, mention in enumerate(candidates):
        span  = mention.get("span", "")
        mtype = mention.get("mention_type", "CLASS")[0]   # C or D for compact display
        print(f"  [{i+1:2d}/{len(candidates)}] [{mtype}] '{span}'", end="  ")
        annotation = pass2_map_mention(mention, text)
        conf    = annotation.get("confidence", 0.0)
        concept = annotation.get("concept", "?")
        inst    = annotation.get("instance") or "—"
        print(f"→ {concept:22s} | inst: {inst:25s} | conf={conf:.2f}")
        if conf >= confidence_threshold and concept not in ("NoMatch", ""):
            annotations.append(annotation)

    print(
        f"\n  Kept {len(annotations)} / {len(candidates)} annotations "
        f"(threshold={confidence_threshold})"
    )
    return annotations

In [7]:
import html as html_lib


def build_highlighted_html(text: str, annotations: list) -> str:
    """Render text with colour-coded highlights by ontology CONCEPT (class)."""
    valid = [
        a for a in annotations
        if a.get("concept", "NoMatch") not in ("NoMatch", "")
        and a.get("start_char", 0) < a.get("end_char", 0)
    ]

    valid.sort(key=lambda a: (a["start_char"], -(a["end_char"] - a["start_char"])))

    kept: list = []
    occupied: set = set()
    for ann in valid:
        positions = set(range(ann["start_char"], ann["end_char"]))
        if not positions & occupied:
            kept.append(ann)
            occupied |= positions

    kept.sort(key=lambda a: a["start_char"])

    parts: list = []
    cursor = 0
    for ann in kept:
        s, e    = ann["start_char"], ann["end_char"]
        concept = ann.get("concept", "")
        color   = CONCEPT_COLORS.get(concept, "#7f8c8d")
        inst    = ann.get("instance")
        conf    = ann.get("confidence", 0.0)
        tooltip = f"{concept}" + (f" → {inst}" if inst else "") + f" | conf={conf:.2f}"

        if s > cursor:
            parts.append(html_lib.escape(text[cursor:s]))

        parts.append(
            f'<mark style="background:{color};color:white;border-radius:4px;'
            f'padding:1px 6px;margin:0 1px;font-weight:bold;cursor:help;" '
            f'title="{html_lib.escape(tooltip)}">'
            f'{html_lib.escape(text[s:e])}'
            f'<sup style="font-size:0.7em;margin-left:2px;">{html_lib.escape(concept)}</sup>'
            f'</mark>'
        )
        cursor = e

    if cursor < len(text):
        parts.append(html_lib.escape(text[cursor:]))

    used_concepts = sorted({a.get("concept", "") for a in kept if a.get("concept")})
    legend = "".join(
        f'<span style="background:{CONCEPT_COLORS.get(c, "#7f8c8d")};color:white;'
        f'padding:2px 8px;border-radius:4px;margin:2px;font-size:0.82em;">{c}</span>'
        for c in used_concepts
    )

    return (
        f'<div style="font-family:Georgia,serif;line-height:2.0;padding:18px;'
        f'border:1px solid #ddd;border-radius:8px;background:#fafafa;white-space:pre-wrap;">'
        f'<div style="margin-bottom:10px;"><b>Concepts found:</b>&nbsp;'
        f'{legend or "<i>none</i>"}</div>'
        f'<hr style="margin:8px 0;border-color:#ddd;">'
        f'{"".join(parts)}'
        f'</div>'
    )


def display_annotations_table(annotations: list, document_label: str = "") -> None:
    """Display concept normalization results as a styled pandas DataFrame."""
    if not annotations:
        print("No annotations to display.")
        return

    rows = [
        {
            "Text Span":       a.get("span", ""),
            "Concept (Class)": a.get("concept", ""),
            "Instance Match":  a.get("instance") or "—",
            "Confidence":      round(a.get("confidence", 0.0), 3),
            "Reasoning":       (
                a.get("reasoning", "")[:90]
                + ("…" if len(a.get("reasoning", "")) > 90 else "")
            ),
        }
        for a in sorted(annotations, key=lambda x: x.get("start_char", 0))
    ]

    df = pd.DataFrame(rows)
    if document_label:
        display(HTML(f"<h4>Concept Normalization Results — {document_label}</h4>"))
    styled = (
        df.style
        .set_properties(**{"text-align": "left", "font-size": "13px"})
        .bar(subset=["Confidence"], color="#2980b9", vmin=0, vmax=1)
        .set_caption(f"{len(rows)} annotations")
    )
    display(styled)


def display_pipeline_results(text: str, annotations: list, document_label: str = "document") -> None:
    display(HTML(f"<h3>{document_label}</h3>"))
    display(HTML("<h4>Highlighted Text</h4>"))
    display(HTML(build_highlighted_html(text, annotations)))
    display(HTML("<h4>Concept Normalization Table</h4>"))
    display_annotations_table(annotations, document_label)

In [ ]:
## Document analysis

file_path = _NOTEBOOK_DIR / "agricola_review.txt"

if file_path.exists():
    file_text   = file_path.read_text(encoding="utf-8")
    annotations = run_two_pass_pipeline(file_text, document_label=file_path.name)
    display_pipeline_results(file_text, annotations, document_label=file_path.name)

    # Add to all_results so the summary cell includes this document
    if "all_results" not in dir():
        all_results = {}
    all_results[file_path.name] = {"text": file_text, "annotations": annotations}
else:
    print(f"File '{file_path}' not found. Place it next to the notebook and update file_path.")

In [9]:
SAMPLE_DOCUMENTS = {
    "game_review": textwrap.dedent("""\
        I have been playing Pandemic with my family for years. Matt Leacock created
        a masterpiece of cooperative game design. It is published by Z-Man Games and
        uses hand management and area movement mechanics. The suggested age is 8+,
        playing time around 45 minutes. Compared to Catan — designed by Klaus Teuber
        and published by KOSMOS — Pandemic feels more thematic and strategic rather
        than economic. My rating for Pandemic is 8 out of 10.
    """).strip(),

    "rulebook_snippet": textwrap.dedent("""\
        CATAN (formerly The Settlers of Catan) is a competitive board game for 3-4
        players. Designed by Klaus Teuber and first released in 1995. It uses dice
        rolling and income mechanics. KOSMOS and Asmodee are among its publishers.
        Complexity rating: 2.28, minimum playing time: 60 minutes. CATAN is a Family
        game type in the Dice and Economic categories. It is a CompetitiveGame.
    """).strip(),

    "expansion_overview": textwrap.dedent("""\
        The Nemesis universe by Awaken Realms, designed by Pawel Samborski, has grown
        into a remarkable series. Nemesis itself is a semi-cooperative survival horror
        experience with hidden traitor mechanics. Nemesis Lockdown introduced variable
        player powers and player elimination. The Carnomorphs expansion adds new enemy
        types. Complexity ranges from 3.32 to 3.91, ratings above 8.0. These games
        fall into the Thematic and Strategic type categories. Solo-friendly with a
        minimum player count of 1.
    """).strip(),
}

print("Sample documents:")
for name, text in SAMPLE_DOCUMENTS.items():
    print(f"  {name}: {len(text)} chars")

Sample documents:
  game_review: 434 chars
  rulebook_snippet: 373 chars
  expansion_overview: 491 chars


In [10]:
all_results = {}

for doc_name, doc_text in SAMPLE_DOCUMENTS.items():
    annotations = run_two_pass_pipeline(
        text=doc_text,
        document_label=doc_name,
        confidence_threshold=0.3,
    )
    all_results[doc_name] = {"text": doc_text, "annotations": annotations}

print("\nAll documents processed.")


Processing: game_review
[ Pass 1 ] Extracting candidate mentions...
  Pass 1 — 11 raw mentions from model
  Pass 1: 11 candidate mentions found

[ Pass 2 ] Concept normalization for 11 mentions...
  [ 1/11] [C] 'Pandemic'  → CooperativeGame        | inst: NEW                       | conf=0.95
  [ 2/11] [C] 'Matt Leacock'  → Designer               | inst: Matt_Leacock              | conf=1.00
  [ 3/11] [C] 'Z-Man Games'  → Publisher              | inst: Z-Man_Games               | conf=1.00
  [ 4/11] [C] 'hand management'  → GameMechanics          | inst: —                         | conf=1.00
  [ 5/11] [C] 'area movement'  → GameMechanics          | inst: AreaMovement              | conf=1.00
  [ 6/11] [D] '8+'  → suggestedAge           | inst: —                         | conf=1.00
  [ 7/11] [D] '45 minutes'  → playingTime            | inst: —                         | conf=1.00
  [ 8/11] [C] 'Catan'  → CompetitiveGame        | inst: CATAN                     | conf=1.00
  [ 9/11] [C] 

In [11]:
for doc_name, result in all_results.items():
    display_pipeline_results(
        text=result["text"],
        annotations=result["annotations"],
        document_label=doc_name,
    )
    display(HTML("<hr style='margin:28px 0;border-color:#ccc;'>"))

,Text Span,Concept (Class),Instance Match,Confidence,Reasoning
0,Pandemic,CooperativeGame,NEW,0.950000,"The mention 'Pandemic' refers to a specific game instance, not a data property or generic …"
1,Matt Leacock,Designer,Matt_Leacock,1.000000,The mention 'Matt Leacock' refers to a specific designer who is listed in the ontology as …
2,Z-Man Games,Publisher,Z-Man_Games,1.000000,The mention 'Z-Man Games' is a named individual in the ontology under the class Publisher.
3,hand management,GameMechanics,—,1.000000,The mention 'hand management' is a generic game mechanic and matches the GameMechanics cla…
4,area movement,GameMechanics,AreaMovement,1.000000,"The mention 'area movement' is a specific game mechanic, which maps to the class GameMecha…"
5,8+,suggestedAge,—,1.000000,"The mention '8+' is a data property value for suggested age, as indicated by the context m…"
6,45 minutes,playingTime,—,1.000000,The mention '45 minutes' is a duration that fits the data property 'bgo:playingTime'. The …
7,Catan,CompetitiveGame,CATAN,1.000000,"The mention 'Catan' refers to the board game CATAN, which is listed as a CompetitiveGame i…"
8,Klaus Teuber,Designer,Klaus_Teuber,1.000000,The mention 'Klaus Teuber' refers to a specific designer listed in the ontology as 'bgo:Kl…
9,KOSMOS,Publisher,KOSMOS,1.000000,"The mention 'KOSMOS' in the context clearly refers to a publisher, and it matches exactly …"


,Text Span,Concept (Class),Instance Match,Confidence,Reasoning
0,CATAN,CompetitiveGame,—,0.950000,"The mention 'CATAN' is a specific game instance, but the hint provided ('birthYear') does …"
1,The Settlers of Catan,CompetitiveGame,—,0.950000,"The mention 'The Settlers of Catan' is a specific game instance, but it does not match any…"
2,Klaus Teuber,Designer,Klaus_Teuber,1.000000,The mention 'Klaus Teuber' is a named individual in the ontology under the class 'bgo:Desi…
3,1995,releaseYear,—,1.000000,The mention '1995' is a specific year that fits the data property 'releaseYear'. There's n…
4,income,GameMechanics,—,1.000000,"The mention 'income' refers to a specific game mechanic, which maps directly to the GameMe…"
5,KOSMOS,Publisher,KOSMOS,1.000000,"The mention 'KOSMOS' directly refers to a publisher of the game CATAN, as indicated by the…"
6,Asmodee,Publisher,asmodee,1.000000,"The mention 'Asmodee' refers to a publisher of board games, matching exactly with the name…"
7,2.28,complexity,—,1.000000,The mention '2.28' corresponds to the data property 'bgo:complexity'. The context clearly …
8,60 minutes,minPlayingTime,—,1.000000,The mention '60 minutes' fits the data property minPlayingTime as it represents the minimu…
9,Family,GameType,—,1.000000,"The mention 'Family' refers to a game type, which aligns with the GameType class in the on…"


,Text Span,Concept (Class),Instance Match,Confidence,Reasoning
0,Nemesis,CooperativeGame,NEW,0.950000,"The mention 'Nemesis' refers to the game title, which is an instance of a CooperativeGame …"
1,Awaken Realms,Publisher,Awaken_Realms,1.000000,"The mention 'Awaken Realms' refers to the publisher of games, and it matches exactly with …"
2,Pawel Samborski,Designer,Paweł_Samborski,1.000000,The mention 'Pawel Samborski' matches the named individual 'bgo:Paweł_Samborski' in the on…
3,hidden traitor mechanics,GameMechanics,—,1.000000,The mention 'hidden traitor mechanics' refers to a specific game mechanic concept within t…
4,Nemesis Lockdown,CooperativeGame,NEW,0.850000,The mention 'Nemesis Lockdown' is described as a variant of the Nemesis game series with s…
5,player elimination,GameMechanics,—,1.000000,"The mention 'player elimination' is a specific game mechanic, which maps to the GameMechan…"
6,Carnomorphs,Expansion,NEW,0.750000,The mention 'Carnomorphs' is described as an expansion that adds new enemy types to the Ne…
7,3.32 to 3.91,complexity,—,1.000000,"The mention '3.32 to 3.91' is clearly referring to the complexity range of a game, which m…"
8,1,minCount,—,1.000000,"The mention '1' in the context refers to the minimum player count, which is a data propert…"
9,8.0,rating,—,1.000000,The mention '8.0' is clearly a data value for the rating property as indicated by the cont…


In [12]:
summary_rows = []
_DATA_PROPS   = set(_dprop_names)
_GAME_CLASSES = _subclass_closure("BoardGame")

for doc_name, result in all_results.items():
    anns    = result["annotations"]
    by_type = defaultdict(int)
    for a in anns:
        by_type[a.get("concept", "Unknown")] += 1
    avg_conf = sum(a.get("confidence", 0.0) for a in anns) / len(anns) if anns else 0.0
    summary_rows.append({
        "Document":       doc_name,
        "Total":          len(anns),
        "BoardGame":      sum(by_type[c] for c in _GAME_CLASSES),
        "Mechanics/Cat":  by_type.get("GameMechanics", 0) + by_type.get("GameCategory", 0)
                          + by_type.get("GameType", 0),
        "Person/Pub":     by_type.get("Designer", 0) + by_type.get("Person", 0)
                          + by_type.get("Publisher", 0),
        "DataProperty":   sum(by_type[p] for p in _DATA_PROPS),
        "Avg Confidence": round(avg_conf, 3),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Document")
display(HTML("<h3>Summary Statistics</h3>"))
display(
    summary_df.style
    .bar(subset=["Total"], color="#2980b9")
    .bar(subset=["Avg Confidence"], color="#27ae60", vmin=0, vmax=1)
    .format({"Avg Confidence": "{:.3f}"})
)

all_anns = [a for r in all_results.values() for a in r["annotations"]]
print(f"\nGrand total annotations: {len(all_anns)}")
concept_counts: dict = defaultdict(int)
for a in all_anns:
    concept_counts[a.get("concept", "Unknown")] += 1
print("\nBreakdown by concept:")
for concept, cnt in sorted(concept_counts.items(), key=lambda x: -x[1]):
    bar = "█" * cnt
    print(f"  {concept:22s}: {cnt:3d}  {bar}")

,Total,BoardGame,Mechanics/Cat,Person/Pub,DataProperty,Avg Confidence
Document,,,,,,
game_review,11,2,2,4,3,0.995
rulebook_snippet,12,2,4,3,3,0.992
expansion_overview,11,3,3,2,3,0.959



Grand total annotations: 34

Breakdown by concept:
  Publisher             :   5  █████
  GameMechanics         :   5  █████
  Designer              :   4  ████
  CooperativeGame       :   3  ███
  CompetitiveGame       :   3  ███
  rating                :   2  ██
  complexity            :   2  ██
  GameType              :   2  ██
  GameCategory          :   2  ██
  suggestedAge          :   1  █
  playingTime           :   1  █
  releaseYear           :   1  █
  minPlayingTime        :   1  █
  Expansion             :   1  █
  minCount              :   1  █
